In [1]:
!pip install pyspark

In [43]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, datediff, substring
from pyspark.ml.feature import VectorAssembler, MinMaxScaler, PCA, StringIndexer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
spark = SparkSession.builder.getOrCreate()

In [6]:
df_video = spark.read.option('header', 'true').parquet('videos-comments-tratados.snappy.parquet')

In [7]:
df_video.show()
df_video.printSchema()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Whenever I go to ...|        0|            8|
|wAZZ-UWGVHI|

In [21]:
mes_publicado = df_video.withColumn('Month', substring(col('Published At'), 6, 2))
mes_publicado.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Month|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|   08|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|   08|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|   08|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Whenever I go to ...|  

In [23]:
indexador = StringIndexer(inputCol = 'Keyword', outputCol = 'Keyword Index')
df_video_indexado = indexador.fit(mes_publicado).transform(mes_publicado)

df_video_indexado.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Month|Keyword Index|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|   08|         17.0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|   08|         17.0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|   08|         17.0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|

In [31]:
df_video_tratado = df_video_indexado.\
    withColumn('Year', col('Year').cast('integer')).\
    withColumn('Month', col('Month').cast('integer'))

df_video_tratado.printSchema()

root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = false)



In [34]:
montar_vetor = VectorAssembler(
    inputCols = ['Likes', 'Views', 'Year', 'Month', 'Keyword Index'],
    outputCol = 'Features'
)

df_video_final = montar_vetor.transform(df_video_tratado)
df_video_final.show()
df_video_final.printSchema()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+--------------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Month|Keyword Index|            Features|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+--------------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|    8|         17.0|[3407.0,135612.0,...|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|    8|         17.0|[3407.0,135612.0,...|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|

In [36]:
df_video_tratado = df_video_final.na.drop()

In [37]:
scaler = MinMaxScaler(inputCol = 'Features', outputCol = 'Features Normal')
df_video_normalizado = scaler.fit(df_video_tratado).transform(df_video_tratado)
df_video_normalizado.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+--------------------+--------------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Month|Keyword Index|            Features|     Features Normal|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+--------------------+--------------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|    8|         17.0|[3407.0,135612.0,...|[2.07229197864298...|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|    8|         17.0|[3407.0,135612.0,...|[2.0722

In [39]:
pca = PCA(k = 1, inputCol = 'Features Normal', outputCol = 'Features PCA')
df_video_pca = pca.fit(df_video_normalizado).transform(df_video_normalizado)
df_video_pca.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+--------------------+--------------------+--------------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Month|Keyword Index|            Features|     Features Normal|        Features PCA|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+--------------------+--------------------+--------------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|    8|         17.0|[3407.0,135612.0,...|[2.07229197864298...| [0.446334517275614]|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in N

In [40]:
train_data, test_data = df_video_pca.randomSplit([0.8, 0.2], seed = 42)

print(f'Treino: {train_data.count()}, Teste: {test_data.count()}')

Treino: 12114, Teste: 2956


In [44]:
regressao_linear = LinearRegression(featuresCol = 'Features Normal', labelCol = 'Comments')
modelo_lr = regressao_linear.fit(train_data)

avaliar_test = modelo_lr.evaluate(test_data)
print('RMSE (Erro Quadrático Médio da Raíz):', avaliar_test.rootMeanSquaredError)
print('R2:(Coeficiente de Determinação):', avaliar_test.r2)

RMSE (Erro Quadrático Médio da Raíz): 15696.328970507317
R2:(Coeficiente de Determinação): 0.8423252962415371


In [45]:
df_video_pca.write.mode('overwrite').option('header', 'true').parquet('videos-preparados-parquet.parquet')

In [46]:
spark.stop()